<p style="text-align:center">
    <a href="https://skills.network" target="_blank">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="200" alt="Skills Network Logo">
    </a>
</p>


<h1>Lab: Data Augmentation and Batching Strategies using PyTorch</h1> 


Estimated time needed: **30** minutes


<h2>Learning Objectives</h2> In this lab, you will:<ul>
<li>Implement text data augmentation techniques such as synonym replacement and random insertion</li><li> Apply token-level transformations to generate diverse training samples</li><li> Create a custom PyTorch Dataset and DataLoader for NLP tasks</li><li> Implement batching with padding and truncation for variable-length sequences</li><li> Compare static vs dynamic batching strategies for performance optimization</li></ul> 


## Table of contents
In this lab, you will augment text data and create efficient PyTorch batching strategies for variable-length NLP inputs.

<ol>
<li><a href="#Import-the-Libraries">Import the Libraries</a> 
</li>
<li><a href="#Build-an-NLP-data-pipeline-in-PyTorch">Build an NLP data pipeline in PyTorch</a>     
</li>
<li><a href="#Create-a-text-corpus">Create a text corpus</a>     
</li>
 <li><a href="#Token-level-transformations">Token-level transformations</a>
</li>
<li><a href="#Synonym-replacement-for-data-augmentation">Synonym replacement for data augmentation</a>     
<li><a href="#Random-insertion-for-text-augmentation">Random insertion for text augmentation</a>  
<li><a href="#Create-a-custom-Dataset">Create a custom Dataset</a> 
</li>
<li><a href="#Static-padding-and-truncation">Static padding and truncation</a> 
</li>
<li><a href="#Dynamic-padding-with-DataLoader">Dynamic padding with DataLoader</a> 
</li>
<li><a href="#Compare-static-and-dynamic-batching">Compare static and dynamic batching</a> 
</li>

</li>
</ol>


## Import the Libraries

Use the code below to import the libraries


In [1]:
# Install PyTorch

!pip install torch

In [2]:
# Import the libraries we need for this lab

import random
import re
from collections import Counter

import torch
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import Dataset, DataLoader

random.seed(2)
torch.manual_seed(2)

<!--Empty Space for separating topics-->


## Build an NLP data pipeline in PyTorch


Data augmentation expands a training set by creating controlled variations of existing samples. For NLP, those variations can include synonym replacement, random insertion, deletion, or swapping while keeping the original label unchanged.


In this lab, the same idea is applied to text by generating new token sequences, encoding them, and using PyTorch batching utilities to prepare efficient model inputs.


<!--Empty Space for separating topics-->


## Create a text corpus


Create a small labeled text corpus where each item contains a sentence and a sentiment label.


In [3]:
# Create a small labeled text corpus

corpus = [
    ("The movie was enjoyable and inspiring", 1),
    ("The product quality was poor and disappointing", 0),
    ("The service was fast and friendly", 1),
    ("The app crashed during checkout", 0),
    ("The lesson was clear and useful", 1),
    ("The package arrived late and damaged", 0),
]

for text, label in corpus:
    print(label, text)

1 The movie was enjoyable and inspiring
0 The product quality was poor and disappointing
1 The service was fast and friendly
0 The app crashed during checkout
1 The lesson was clear and useful
0 The package arrived late and damaged


Separate the text samples and labels:


In [4]:
# Split texts and labels

texts = [item[0] for item in corpus]
labels = [item[1] for item in corpus]

print(texts[:2])
print(labels[:2])

['The movie was enjoyable and inspiring', 'The product quality was poor and disappointing']
[1, 0]


<!--Empty Space for separating topics-->


## Token-level transformations


Token-level transformations operate on words or subwords. First, create a simple tokenizer that converts text to lowercase and removes punctuation.


In [5]:
# Create a tokenizer

def tokenize(text):
    return re.findall(r"\b\w+\b", text.lower())

Tokenize the text corpus:


In [6]:
# Tokenize all texts

tokenized_texts = [tokenize(text) for text in texts]

for tokens in tokenized_texts:
    print(tokens)

['the', 'movie', 'was', 'enjoyable', 'and', 'inspiring']
['the', 'product', 'quality', 'was', 'poor', 'and', 'disappointing']
['the', 'service', 'was', 'fast', 'and', 'friendly']
['the', 'app', 'crashed', 'during', 'checkout']
['the', 'lesson', 'was', 'clear', 'and', 'useful']
['the', 'package', 'arrived', 'late', 'and', 'damaged']


<!--Empty Space for separating topics-->


## Synonym replacement for data augmentation


Synonym replacement creates a new training example by replacing selected tokens with similar words.


In [7]:
# Create a small synonym dictionary

synonyms = {
    "enjoyable": ["pleasant", "fun"],
    "inspiring": ["motivating", "uplifting"],
    "poor": ["bad", "weak"],
    "disappointing": ["unsatisfying", "frustrating"],
    "fast": ["quick", "speedy"],
    "friendly": ["kind", "helpful"],
    "clear": ["simple", "understandable"],
    "useful": ["helpful", "practical"],
    "late": ["delayed", "slow"],
    "damaged": ["broken", "defective"],
}

Define a function that replaces one eligible token with a synonym:


In [8]:
# Replace one token with a synonym

def synonym_replacement(tokens, synonym_map):
    new_tokens = tokens.copy()
    candidates = [index for index, token in enumerate(tokens) if token in synonym_map]
    if not candidates:
        return new_tokens
    selected_index = random.choice(candidates)
    selected_token = tokens[selected_index]
    new_tokens[selected_index] = random.choice(synonym_map[selected_token])
    return new_tokens

Apply synonym replacement to a sample sentence:


In [9]:
# Apply synonym replacement

example_tokens = tokenized_texts[0]
augmented_tokens = synonym_replacement(example_tokens, synonyms)

print("Original:", example_tokens)
print("Augmented:", augmented_tokens)

Original: ['the', 'movie', 'was', 'enjoyable', 'and', 'inspiring']
Augmented: ['the', 'movie', 'was', 'pleasant', 'and', 'inspiring']


<!--Empty Space for separating topics-->


## Random insertion for text augmentation


Random insertion adds a related token into the sequence. This can make the model less sensitive to small wording changes.


In [10]:
# Insert a synonym at a random position

def random_insertion(tokens, synonym_map):
    new_tokens = tokens.copy()
    candidates = [token for token in tokens if token in synonym_map]
    if not candidates:
        return new_tokens
    selected_token = random.choice(candidates)
    inserted_token = random.choice(synonym_map[selected_token])
    insert_position = random.randint(0, len(new_tokens))
    new_tokens.insert(insert_position, inserted_token)
    return new_tokens

Apply random insertion to a sample sentence:


In [11]:
# Apply random insertion

inserted_tokens = random_insertion(example_tokens, synonyms)

print("Original:", example_tokens)
print("Inserted:", inserted_tokens)

Original: ['the', 'movie', 'was', 'enjoyable', 'and', 'inspiring']
Inserted: ['the', 'movie', 'was', 'enjoyable', 'and', 'inspiring', 'fun']


Create an augmented dataset by combining the original examples with transformed examples that use the same labels.


In [12]:
# Generate augmented training samples

augmented_samples = []

for tokens, label in zip(tokenized_texts, labels):
    augmented_samples.append((tokens, label))
    augmented_samples.append((synonym_replacement(tokens, synonyms), label))
    augmented_samples.append((random_insertion(tokens, synonyms), label))

print("Original samples:", len(corpus))
print("Augmented samples:", len(augmented_samples))
print(augmented_samples[:3])

Original samples: 6
Augmented samples: 18
[(['the', 'movie', 'was', 'enjoyable', 'and', 'inspiring'], 1), (['the', 'movie', 'was', 'fun', 'and', 'inspiring'], 1), (['the', 'movie', 'was', 'enjoyable', 'motivating', 'and', 'inspiring'], 1)]


<!--Empty Space for separating topics-->


## Create a custom Dataset


Build a vocabulary from the augmented tokens, then convert each token sequence into token IDs.


In [13]:
# Build a vocabulary

special_tokens = ["<pad>", "<unk>"]
token_counts = Counter(token for tokens, label in augmented_samples for token in tokens)
vocab = {token: index for index, token in enumerate(special_tokens)}

for token in token_counts:
    if token not in vocab:
        vocab[token] = len(vocab)

pad_id = vocab["<pad>"]
unk_id = vocab["<unk>"]

print("Vocabulary size:", len(vocab))
print(list(vocab.items())[:10])

Vocabulary size: 34
[('<pad>', 0), ('<unk>', 1), ('the', 2), ('movie', 3), ('was', 4), ('enjoyable', 5), ('and', 6), ('inspiring', 7), ('fun', 8), ('motivating', 9)]


Create helper functions to encode, pad, and truncate sequences:


In [14]:
# Encode and pad sequences

def encode_tokens(tokens, vocabulary):
    return [vocabulary.get(token, unk_id) for token in tokens]

def pad_or_truncate(sequence, max_length, padding_id):
    if len(sequence) > max_length:
        return sequence[:max_length]
    return sequence + [padding_id] * (max_length - len(sequence))

Define a custom PyTorch Dataset for NLP samples:


In [15]:
# Create a custom Dataset

class TextAugmentationDataset(Dataset):
    def __init__(self, samples, vocabulary):
        self.samples = samples
        self.vocabulary = vocabulary

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        tokens, label = self.samples[index]
        token_ids = encode_tokens(tokens, self.vocabulary)
        return torch.tensor(token_ids, dtype=torch.long), torch.tensor(label, dtype=torch.long)


dataset = TextAugmentationDataset(augmented_samples, vocab)

print("Dataset size:", len(dataset))
print(dataset[0])

Dataset size: 18
(tensor([2, 3, 4, 5, 6, 7]), tensor(1))


<!--Empty Space for separating topics-->


## Static padding and truncation


Static batching pads or truncates all sequences to the same fixed maximum length before batching.


In [16]:
# Create a collate function for static batching

def static_collate_batch(batch, max_length=10):
    sequences, labels = zip(*batch)
    padded_sequences = [pad_or_truncate(sequence.tolist(), max_length, pad_id) for sequence in sequences]
    return torch.tensor(padded_sequences, dtype=torch.long), torch.stack(labels)

Create a DataLoader that uses static padding:


In [17]:
# Load a static padded batch

static_loader = DataLoader(dataset, batch_size=4, shuffle=False, collate_fn=static_collate_batch)

static_features, static_labels = next(iter(static_loader))

print(static_features)
print("Feature shape:", static_features.shape)
print("Labels:", static_labels)

tensor([[ 2,  3,  4,  5,  6,  7,  0,  0,  0,  0],
        [ 2,  3,  4,  8,  6,  7,  0,  0,  0,  0],
        [ 2,  3,  4,  5,  9,  6,  7,  0,  0,  0],
        [ 2, 10, 11,  4, 12,  6, 13,  0,  0,  0]])
Feature shape: torch.Size([4, 10])
Labels: tensor([1, 1, 1, 0])


<!--Empty Space for separating topics-->


## Dynamic padding with DataLoader


Dynamic batching pads each batch only to the longest sequence in that batch. This can reduce unnecessary padding.


In [18]:
# Create a collate function for dynamic batching

def dynamic_collate_batch(batch):
    sequences, labels = zip(*batch)
    padded_sequences = pad_sequence(sequences, batch_first=True, padding_value=pad_id)
    return padded_sequences, torch.stack(labels)

Create a DataLoader that uses dynamic padding:


In [19]:
# Load a dynamic padded batch

dynamic_loader = DataLoader(dataset, batch_size=4, shuffle=False, collate_fn=dynamic_collate_batch)

dynamic_features, dynamic_labels = next(iter(dynamic_loader))

print(dynamic_features)
print("Feature shape:", dynamic_features.shape)
print("Labels:", dynamic_labels)

tensor([[ 2,  3,  4,  5,  6,  7,  0],
        [ 2,  3,  4,  8,  6,  7,  0],
        [ 2,  3,  4,  5,  9,  6,  7],
        [ 2, 10, 11,  4, 12,  6, 13]])
Feature shape: torch.Size([4, 7])
Labels: tensor([1, 1, 1, 0])


<!--Empty Space for separating topics-->


## Compare static and dynamic batching


Compare the number of padding tokens produced by static and dynamic batching. Fewer padding tokens usually means less wasted computation.


In [20]:
# Count padding tokens

def count_padding(batch_features, padding_id):
    return int((batch_features == padding_id).sum().item())

static_padding = count_padding(static_features, pad_id)
dynamic_padding = count_padding(dynamic_features, pad_id)

print("Static padding tokens:", static_padding)
print("Dynamic padding tokens:", dynamic_padding)

Static padding tokens: 14
Dynamic padding tokens: 2


Static padding provides predictable tensor shapes, while dynamic padding can improve efficiency when sequence lengths vary widely.


In [ ]:
# Inspect all dynamic batch shapes

for batch_features, batch_labels in dynamic_loader:
    print(batch_features.shape)


<a href="https://dataplatform.cloud.ibm.com/registration/stepone?utm_source=skills_network&utm_content=in_lab_content_link&utm_id=Lab-IBMDeveloperSkillsNetwork-DL0110EN-SkillsNetwork&context=cpdaas&apps=data_science_experience%2Cwatson_machine_learning"><img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DL0110EN-SkillsNetwork/Template/module%201/images/Watson_Studio.png"></a>


<!--Empty Space for separating topics-->


<h2>About the Author:</h2> 

<b> Harish Pant </b> is an accomplished IT Consultant specializing in bridging the gap between complex AI technologies and practical business applications. Holding a Bachelor’s degree in Computer Science with a strong background in Machine Learning, Harish serves clients across diverse industries—including fintech, retail, and manufacturing—helping them navigate AI adoption. Harish is known for translating technical AI concepts into actionable business strategies and delivering engaging, high-impact presentations.


<!--
## Change Log

|  Date (YYYY-MM-DD) |  Version | Changed By  |  Change Description |
|---|---|---|---|
| 2026-05-10  | 1.0  | IBM Skills Network  |  Created lab for data augmentation and batching strategies using PyTorch |
| 2026-05-10  | 2.0  | Jojy JA |  Reviewed the lab |

-->


<hr>





## <h3 align="center"> &#169; IBM Corporation. All rights reserved. <h3/>
